# Grid Based Declustering

This notebook demonstrates grid based declustering approaches using inverse distance weighting and K-nearest neighbors. 

It is assumed that prior to running this example notebook a Seequent Evo workspace has been created and populated with the necessary example data. If not, refer to the prior notebook [1-populate-workspace.ipynb`](./1-populate-workspace.ipynb)

Like any Python based workflow leveraging Seequent Evo, the first step is to import the necessary packages and authenticate with Seequent Evo such that data can be accessed.

In [ ]:
import numpy as np
import plotly.express as px
import plotly.io as pio

import evo.compute.tasks as evo_tasks
import evo.compute.tasks.geostatistics.declustering as declus
import evo.objects.typed as evo_objs
from evo.notebooks import ServiceManagerWidget

pio.templates.default = "simple_white"
%load_ext evo.widgets

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(client_id="<CLIENT-ID>").login()

## Load the Objects Needed

Note that when accessing an object from the Evo workspace, a reference to that object is returned.

This reference contains information about the unique ID and location of that object as well as the coordinates and attributes. What this object referece does **not** contain is the actual data. 

The object reference and the tabluar data of the object are separated such that workflows can be orchestrated on Evo without ever having to download the data locally. However, if access to the underlying data is desired, use the `.to_dataframe` method as needed to access the numerical data associated with the object.

In this case, a pointset and the blockmodel are needed.

In [ ]:
comps = await evo_objs.object_from_path(evo_connection, "WP comps.json")
block_model = await evo_objs.object_from_path(evo_connection, "WP Regular BlockModel.json")

Additional information such as meta data and the list of attributes can be seen using the html widget associated with the object.

In [ ]:
comps

## Set up Declustering Parameters

Each Geostatistics compute tasks accepts a series of inputs and parameters. Details of each are available in [the devloper portal](https://developer.seequent.com/docs/api/geostatistics-task/geostatistics-task-api).

For this example, first prepare parameters for inverse distance weighted declustering.

In [ ]:
# first create the search neighborhood
ellipsoid = evo_tasks.Ellipsoid(ranges=evo_tasks.EllipsoidRanges(200, 200, 200), rotation=evo_tasks.Rotation(0, 0, 0))
srch = evo_tasks.SearchNeighborhood(ellipsoid=ellipsoid, min_samples=1, max_samples=10)


# next, establish the rest of the parameters
declus_params = declus.DeclusteringParameters(
    source=comps, grid=block_model, target=comps.attributes["IDW declus"], neighborhood=srch, power=2.0
)

## Execute the Compute Task
Once parameters have been established, execute the task using the `.run` method. It is important to note that neither the composites, nor the block model were ever downloaded locally. When executing a geostatistical compute task on Seeuent Evo the user is in a position of defining the inputs, outputs and parameters to orchestrate the computational task that will be executed on Evo. 

Lastly, be mindful of the `await` that preceeds the execution of the compute task. This small bit of python code permits running many comptue tasks concurrently in an asyncronous way to maximize efficency when running multiple tasks at once.

In [ ]:
declus_result = await evo_tasks.run(evo_connection, declus_params, preview = True)

## Investigate Results

Because compute tasks are orchestrated by the user and executed on Evo, the final results are saved directly to the Seequent Evo workspace. To investigate the results locally, it is necessary to refesh the local version of the object.

In [ ]:
updated_comps = await evo_objs.object_from_path(evo_connection, "WP comps.json")
updated_comps

The newly created declustering weights can be viewed in the Seequent Evo Viewer by clicking on the `Viewer` link above. Or downloaded locally for statistical evaluation.
To download locally, the `.to_dataframe` method is used with only the necessary columns specified.

In [ ]:
comps_df = await updated_comps.to_dataframe("Cu_pct", "IDW declus")
comps_df.head()

Using [plotly](https://docs.plotly.com/), generate a histogram comparing the original and IDW declustered CU_pct distributions.

In [ ]:
hist = px.histogram(
    comps_df.assign(distribution="Original"),
    x="Cu_pct",
    color="distribution",
    histnorm="probability density",
    color_discrete_sequence=["steelblue"],
)

weighted_hist = px.histogram(
    comps_df.assign(distribution="IDW declustered"),
    x="Cu_pct",
    y="IDW declus",
    color="distribution",
    histfunc="sum",
    histnorm="probability density",
    color_discrete_sequence=["darkorange"],
)

hist.add_trace(weighted_hist.data[0])
hist.update_layout(barmode="overlay", title="Cu_pct Distribution: Original vs IDW Declustered")
hist.show()

Now, examine the weighted statistics. Note that [pandas' `.describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) doesn't provide weighted statistics so leverage [numpy's `.average`](https://numpy.org/doc/stable/reference/generated/numpy.average.html) method for this analysis.

In [ ]:
import pandas as pd


def compare_weighted_summary(values, weights):
    mean = np.average(values, weights=weights)
    return pd.DataFrame(
        {
            "Original": [len(values), values.min(), values.mean(), values.var(ddof=0), values.max()],
            "IDW declustered": [
                len(values),
                values.min(),
                mean,
                np.average((values - mean) ** 2, weights=weights),
                values.max(),
            ],
        },
        index=["count", "min", "mean", "variance", "max"],
    )


compare_weighted_summary(values=comps_df["Cu_pct"], weights=comps_df["IDW declus"])